## Setup

In [42]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor


import mlflow
import dagshub
import mlflow.sklearn

from experiment_DataCleaning_Outliers import threshold



In [43]:
# MLflow
dagshub.init(
    repo_owner='IzaKakhniashvili',
    repo_name='ML-assignment1-HousePrices',
    mlflow=True
)


Initialized MLflow to track repo "IzaKakhniashvili/ML-assignment1-HousePrices"

Repository IzaKakhniashvili/ML-assignment1-HousePrices initialized!

In [44]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [45]:
train.shape, test.shape

((1460, 81), (1459, 80))

In [46]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [47]:
train["SalePrice"].describe()

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

## Ex6_Ordinal Encoding

#### Data Cleaning

In [48]:
df_f = train.copy()
df_f = df_f.drop(df_f[(df_f['GrLivArea'] > 4000) & (df_f['SalePrice'] < 300000)].index)

y_f = df_f["SalePrice"]
X_f = df_f.drop("SalePrice", axis=1)

X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

In [49]:
X_tr = X_train_f.copy()
X_v = X_val_f.copy()

In [50]:
missing_pct = X_tr.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.8].index.tolist()
if 'FireplaceQu' in cols_to_drop: cols_to_drop.remove('FireplaceQu')

In [51]:
X_tr = X_tr.drop(columns=cols_to_drop)
X_v = X_v.drop(columns=cols_to_drop)

Ordinal Encoding

In [52]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
existing_qual_cols = [c for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond'] if c in X_tr.columns]

In [53]:
for col in existing_qual_cols:
    X_tr[col] = X_tr[col].fillna('None').map(qual_map)
    X_v[col] = X_v[col].fillna('None').map(qual_map)


#### Feature Engineering

In [54]:
for df in [X_tr, X_v]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['Quality_x_Size'] = df['OverallQual'] * df['GrLivArea']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemod'] = df['YrSold'] - df['YearRemodAdd']

In [55]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Selection

In [56]:
X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [65]:
temp_df = pd.concat([X_tr_enc, y_train_f], axis=1)
threshold =0.2
relevant_cols = temp_df.corr()['SalePrice'][abs(temp_df.corr()['SalePrice']) > 0.2].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]


In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [59]:
y_train_log = np.log1p(y_train_f)
model = LinearRegression()
model.fit(X_tr_scaled, y_train_log)
final_preds = np.expm1(model.predict(X_v_scaled))

In [60]:
from sklearn.metrics import mean_squared_log_error

rmse = np.sqrt(mean_squared_log_error(y_val_f, final_preds))
rmse

np.float64(0.14800758695282362)

In [61]:
r2 = r2_score(y_val_f, final_preds)
r2

0.8995495633761422

In [62]:
mae = mean_absolute_error(y_val_f, final_preds)
mae

16806.883946230544

In [66]:
with mlflow.start_run(run_name="Exp6_Ordinal"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 6")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", 0.2)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/15 21:15:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 21:15:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/15 21:15:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 25
Created version '25' of model 'house_price_model'.


🏃 View run Exp6_Ordinal at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/2b17cd6440844df1ae7634c21e4d0624
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Ex7_RIDGE & Binary Remodel Feature

#### Data Cleaning

In [67]:
df = train.copy()
df = df.drop(df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)].index)

y = df["SalePrice"]
X = df.drop("SalePrice", axis=1)


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [68]:
X_tr = X_train.copy()
X_v = X_val.copy()


In [69]:
missing_pct = X_tr.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.89].index.tolist()
if 'FireplaceQu' in cols_to_drop: cols_to_drop.remove('FireplaceQu') # სასარგებლოა

X_tr = X_tr.drop(columns=cols_to_drop)
X_v = X_v.drop(columns=cols_to_drop)

In [70]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols = [c for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu'] if c in X_tr.columns]

for col in qual_cols:
    X_tr[col] = X_tr[col].fillna('None').map(qual_map)
    X_v[col] = X_v[col].fillna('None').map(qual_map)

#### Feature Engineering

In [71]:
for df_temp in [X_tr, X_v]:
    df_temp['TotalSF'] = df_temp['TotalBsmtSF'] + df_temp['1stFlrSF'] + df_temp['2ndFlrSF']
    df_temp['TotalBath'] = df_temp['FullBath'] + (0.5 * df_temp['HalfBath']) + df_temp['BsmtFullBath']
    df_temp['HouseAge'] = df_temp['YrSold'] - df_temp['YearBuilt']
    df_temp['IsRemodeled'] = (df_temp['YearRemodAdd'] != df_temp['YearBuilt']).astype(int)

In [72]:
skew_cols = ['LotArea', 'GrLivArea', '1stFlrSF', 'TotalBsmtSF']
for col in skew_cols:
    X_tr[col] = np.log1p(X_tr[col])
    X_v[col] = np.log1p(X_v[col])

num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.get_dummies(pd.concat([X_tr, X_v]), columns=cat_cols)
X_tr_enc = combined[combined['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined[combined['is_train'] == 0].drop('is_train', axis=1)

#### Feature Selection

In [73]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
relevant_cols = temp_df.corr()['SalePrice'][abs(temp_df.corr()['SalePrice']) > 0.15].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [74]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [78]:
from sklearn.linear_model import RidgeCV
y_train_log = np.log1p(y_train)
model = RidgeCV(alphas=np.logspace(-2, 2, 20), cv=5)
model.fit(X_tr_scaled, y_train_log)
final_preds = np.expm1(model.predict(X_v_scaled))

In [79]:
rmse = np.sqrt(mean_squared_log_error(y_val, final_preds))
rmse

np.float64(0.13146550902914936)

In [80]:
r2 = r2_score(y_val, final_preds)
r2

0.9148287954168752

In [81]:
mae = mean_absolute_error(y_val_f, final_preds)
mae

15278.959273600985

In [82]:
with mlflow.start_run(run_name="Exp_7"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 7")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", 0.15)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/15 21:17:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/15 21:17:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/15 21:17:43 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 26
Created version '26' of model 'house_price_model'.


🏃 View run Exp_7 at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/96273ae07bec41f5982aaf2adca6a0a5
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Ex8_Neighborhood Ranking

#### Data Cleaning

In [83]:
df_h = train.copy()
df_h = df_h.drop(df_h[(df_h['GrLivArea'] > 4000) & (df_h['SalePrice'] < 300000)].index)

y_h = df_h["SalePrice"]
X_h = df_h.drop("SalePrice", axis=1)

X_train_h, X_val_h, y_train_h, y_val_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

In [84]:
X_tr = X_train_h.copy()
X_v = X_val_h.copy()

In [85]:
temp_df = pd.concat([X_tr, y_train_h], axis=1)
neigh_map = temp_df.groupby('Neighborhood')['SalePrice'].median().sort_values()

In [86]:
bins = pd.qcut(neigh_map, 3, labels=[1, 2, 3]).to_dict()

In [87]:
X_tr['Neigh_Rank'] = X_tr['Neighborhood'].map(bins).astype(float)
X_v['Neigh_Rank'] = X_v['Neighborhood'].map(bins).astype(float).fillna(2.0)

In [88]:
binary_features = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea']
for col in binary_features:
    if col in X_tr.columns:
        X_tr[f'Has_{col}'] = (X_tr[col] > 0).astype(int)
        X_v[f'Has_{col}'] = (X_v[col] > 0).astype(int)

In [89]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols = ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'FireplaceQu']
for col in qual_cols:
    if col in X_tr.columns:
        X_tr[col] = X_tr[col].fillna('None').map(qual_map)
        X_v[col] = X_v[col].fillna('None').map(qual_map)

#### Feature Engineering

In [90]:
for df_temp in [X_tr, X_v]:
    df_temp['TotalSF'] = df_temp['TotalBsmtSF'] + df_temp['1stFlrSF'] + df_temp['2ndFlrSF']
    df_temp['Total_Qual'] = df_temp['OverallQual'] + df_temp['OverallCond']
    df_temp['GrLivArea'] = np.log1p(df_temp['GrLivArea'])
    df_temp['LotArea'] = np.log1p(df_temp['LotArea'])

In [91]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")


In [92]:
X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.get_dummies(pd.concat([X_tr, X_v]), columns=cat_cols)
X_tr_enc = combined[combined['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined[combined['is_train'] == 0].drop('is_train', axis=1)

#### Feature Selection

In [93]:
from sklearn.preprocessing import StandardScaler
temp_corr = pd.concat([X_tr_enc, y_train_h], axis=1)
relevant_cols = temp_corr.corr()['SalePrice'][abs(temp_corr.corr()['SalePrice']) > 0.12].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)
print(X_tr_final.columns.tolist())

['LotFrontage', 'LotArea', 'OverallQual', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'ExterQual', 'BsmtQual', 'BsmtFinSF1', 'BsmtUnfSF', 'TotalBsmtSF', 'HeatingQC', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'BsmtFullBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Fireplaces', 'FireplaceQu', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'PoolArea', 'Neigh_Rank', 'Has_WoodDeckSF', 'Has_OpenPorchSF', 'Has_EnclosedPorch', 'Has_PoolArea', 'TotalSF', 'Total_Qual', 'MSZoning_RL', 'MSZoning_RM', 'Alley_Grvl', 'Alley_None', 'LotShape_IR1', 'LotShape_IR2', 'LotShape_Reg', 'LotConfig_CulDSac', 'Neighborhood_BrkSide', 'Neighborhood_Edwards', 'Neighborhood_IDOTRR', 'Neighborhood_NAmes', 'Neighborhood_NoRidge', 'Neighborhood_NridgHt', 'Neighborhood_OldTown', 'Neighborhood_Sawyer', 'Neighborhood_Somerst', 'Neighborhood_StoneBr', 'BldgType_1Fam', 'HouseStyle_1.5Fin', 'HouseStyle_2Story', 'RoofStyle_Gable', 'RoofStyle_

In [94]:
from sklearn.linear_model import RidgeCV

y_train_log = np.log1p(y_train_h)
model = RidgeCV(alphas=np.logspace(-1, 2, 30), cv=5)
model.fit(X_tr_scaled, y_train_log)

final_preds = np.expm1(model.predict(X_v_scaled))

In [95]:
rmse = np.sqrt(mean_squared_log_error(y_val_h, final_preds))
rmse

np.float64(0.12319322536448843)

In [96]:
r2 = r2_score(y_val_h, final_preds)
r2

0.921254239340018

In [97]:
mae = mean_absolute_error(y_val_f, final_preds)
mae

14768.330581471459

In [ ]:
with mlflow.start_run(run_name="Exp_8"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 8")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", 0.12)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )